# 1. Introducao

Este notebook faz um diagnostico de **higiene das features** antes de novos refinamentos de modelagem.

Aqui, higiene significa:
- identificar baixa variancia e colunas quase constantes
- identificar correlacoes altas e possiveis redundancias
- entender riscos para modelos lineares e impacto potencial em boosting
- gerar recomendacoes praticas para manter, revisar ou eventualmente excluir colunas

Importante:
- este notebook **nao faz feature selection definitiva**
- a avaliacao e diagnostica e orientada a decisao metodologica
- o alvo observado e `s`, usado apenas como proxy operacional

Modo de execucao:
- `SMOKE=True` para diagnostico rapido e reprodutivel
- `SMOKE=False` para referencia final no dataset completo

# 2. Carregamento dos dados

Nesta etapa, carregamos o dataset consolidado por agricultor (nivel pessoa).

O objetivo estatistico do bloco e garantir que:
- a base de entrada existe e esta acessivel
- o modo smoke reduza custo computacional sem mudar a logica analitica
- as colunas estruturais minimas estejam presentes

In [ ]:
# ---------------------------------------------------------
# Setup reprodutivel e leitura do dataset
# ---------------------------------------------------------
# Contexto:
# Este bloco prepara ambiente, seeds e caminhos para reproducao.
#
# Objetivo estatistico:
# Evitar variacoes nao controladas e garantir leitura consistente da base.
#
# Operacao computacional:
# - Define seed e modo SMOKE
# - Carrega parquet consolidado
# - Aplica amostragem reprodutivel quando SMOKE=True
#
# Interpretacao esperada:
# Ao final, temos um DataFrame valido para diagnosticos de higiene.

import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import roc_auc_score

SEED = 42
SMOKE = False
SMOKE_MAX_ROWS = 300_000
CORR_THRESHOLD = 0.90
DOMINANT_SHARE_THRESHOLD = 0.99

np.random.seed(SEED)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_PATH = ROOT / "dados" / "dados_processados" / "df_caf_demografia_ufpa_pronaf_rotulo_pu.parquet"
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Arquivo ausente: {DATA_PATH}")

t0 = time.perf_counter()
df = pd.read_parquet(DATA_PATH, engine="pyarrow")
read_s = time.perf_counter() - t0
print(f"Shape original: {df.shape}")
print(f"Leitura do parquet: {read_s:.2f}s")

if SMOKE and len(df) > SMOKE_MAX_ROWS:
    df = df.sample(n=SMOKE_MAX_ROWS, random_state=SEED).reset_index(drop=True)
    print(f"[SMOKE] Shape apos amostra: {df.shape}")

print(f"SMOKE={SMOKE} | SMOKE_MAX_ROWS={SMOKE_MAX_ROWS}")


# 3. Definicao do conjunto de features

Separacao explicita de colunas especiais:
- `TARGET_COL`
- chaves de identificacao (`ID_COLS`)
- metadado (`META_COLS`)

Essas colunas **nao entram** na analise de higiene das features do modelo.

In [ ]:
TARGET_COL = "s"
GROUP_COL = "id_familia"
ID_COLS = [
    "cpf_agricultor",
    "id_familia",
]
META_COLS = [
    "ibge",
]

required_cols = set([TARGET_COL, GROUP_COL] + ID_COLS + META_COLS)
missing_required = sorted(required_cols - set(df.columns))
if missing_required:
    raise ValueError(f"Colunas obrigatorias ausentes: {missing_required}")

s_num = pd.to_numeric(df[TARGET_COL], errors="coerce")
if s_num.isna().any() or (~s_num.isin([0, 1])).any():
    raise ValueError("TARGET_COL='s' deve ser binaria {0,1}.")
df[TARGET_COL] = s_num.astype("int8")

feature_cols = [c for c in df.columns if c not in ([TARGET_COL] + ID_COLS + META_COLS)]
numeric_feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df[c])]
non_numeric_feature_cols = [c for c in feature_cols if c not in numeric_feature_cols]

print(f"Total de colunas no dataset: {df.shape[1]}")
print(f"Total de features candidatas: {len(feature_cols)}")
print(f"Features numericas: {len(numeric_feature_cols)}")
print(f"Features nao numericas: {len(non_numeric_feature_cols)}")

if non_numeric_feature_cols:
    print("Features nao numericas detectadas (top 20):")
    display(pd.DataFrame({"feature_nao_numerica": non_numeric_feature_cols[:20]}))


In [ ]:
df.head()

# feature_cols

# 4. Diagnostico de tipos e missing

Objetivo desta secao:
- mapear dtypes das features
- medir cobertura por coluna
- identificar onde missing pode elevar risco de instabilidade em modelagem

Interpretacao:
- missing alto **nao implica exclusao automatica**
- o diagnostico aponta risco e prioridades de revisao

In [ ]:
dtype_df = (
    df[feature_cols]
    .dtypes.astype("string")
    .rename("dtype")
    .to_frame()
    .reset_index()
    .rename(columns={"index": "feature"})
    .sort_values(["dtype", "feature"])
    .reset_index(drop=True)
)

missing_df = (
    df[feature_cols]
    .isna()
    .mean()
    .rename("pct_missing")
    .to_frame()
    .reset_index()
    .rename(columns={"index": "feature"})
)
missing_df["n_missing"] = (missing_df["pct_missing"] * len(df)).round().astype("int64")
missing_df = missing_df.sort_values("pct_missing", ascending=False).reset_index(drop=True)

print("Resumo de dtypes das features:")
display(dtype_df.groupby("dtype", dropna=False)["feature"].count().rename("n_features").to_frame())

print("Top 20 features por percentual de missing:")
display(missing_df.head(20))


# 5. Diagnostico de baixa variancia

Este bloco busca sinais de features com baixo potencial informativo bruto:
- variancia zero
- quase constantes
- poucos valores unicos fora do contexto esperado

Cuidado de interpretacao:
- flags binarias podem ter baixa variancia e ainda serem uteis
- por isso este diagnostico e um filtro de risco, nao decisao final

In [ ]:
def dominant_share(series: pd.Series) -> float:
    vc = series.value_counts(dropna=False, normalize=True)
    if vc.empty:
        return 1.0
    return float(vc.iloc[0])


def is_name_expected_binary(col_name: str) -> bool:
    # Heuristica de nomenclatura para flags binarias no contexto CAF.
    patterns = ("st_", "fl_", "flag_", "tem_", "possui_", "is_")
    return col_name.startswith(patterns) or col_name in {"A", "AC", "B", "V", "Desconhecido"}


variance_series = df[numeric_feature_cols].var(ddof=0)
nunique_series = df[numeric_feature_cols].nunique(dropna=False)
dominant_share_series = df[numeric_feature_cols].apply(dominant_share)

low_var_df = pd.DataFrame(
    {
        "feature": numeric_feature_cols,
        "variance": variance_series.reindex(numeric_feature_cols).to_numpy(),
        "n_unique": nunique_series.reindex(numeric_feature_cols).to_numpy(),
        "dominant_share": dominant_share_series.reindex(numeric_feature_cols).to_numpy(),
    }
)
low_var_df["zero_variance"] = low_var_df["variance"].eq(0.0)
low_var_df["expected_binary_by_name"] = low_var_df["feature"].map(is_name_expected_binary)
low_var_df["very_low_cardinality_non_binary_name"] = (
    low_var_df["n_unique"].le(2) & (~low_var_df["expected_binary_by_name"])
)
low_var_df["dominant_over_99pct"] = low_var_df["dominant_share"].ge(DOMINANT_SHARE_THRESHOLD)
low_var_df["candidate_review"] = (
    low_var_df["zero_variance"]
    | low_var_df["very_low_cardinality_non_binary_name"]
    | low_var_df["dominant_over_99pct"]
)

print("Top 20 menores variancias:")
display(low_var_df.sort_values(["variance", "n_unique"], ascending=[True, True]).head(20))

print("Top 20 maiores concentracoes (dominant_share):")
display(low_var_df.sort_values("dominant_share", ascending=False).head(20))

print("Candidatas a revisao por baixa variancia/concentracao:")
display(
    low_var_df.loc[low_var_df["candidate_review"]]
    .sort_values(["zero_variance", "dominant_share", "n_unique"], ascending=[False, False, True])
    .reset_index(drop=True)
)


# 6. Diagnostico de correlacao entre features

Aqui avaliamos redundancia linear:
- matriz de correlacao das features numericas
- pares com `|corr| >= CORR_THRESHOLD`

Interpretacao:
- correlacao alta pode gerar redundancia e instabilidade em modelos lineares
- em boosting, redundancia pode ser menos critica, mas ainda pode afetar eficiencia e interpretacao

In [ ]:
corr_matrix = df[numeric_feature_cols].corr(method="pearson", numeric_only=True)

upper_mask = np.triu(np.ones(corr_matrix.shape, dtype=bool), k=1)
upper_corr = corr_matrix.where(upper_mask)


def stack_non_na(frame: pd.DataFrame) -> pd.Series:
    # Compatibilidade entre versoes de pandas para stack sem avisos desnecessarios.
    try:
        return frame.stack(future_stack=True).dropna()
    except TypeError:
        return frame.stack(dropna=True)


high_corr_pairs = (
    stack_non_na(upper_corr)
    .rename("corr")
    .reset_index()
    .rename(columns={"level_0": "feature_1", "level_1": "feature_2"})
)
high_corr_pairs["corr_abs"] = high_corr_pairs["corr"].abs()
high_corr_pairs = high_corr_pairs.loc[high_corr_pairs["corr_abs"] >= CORR_THRESHOLD]
high_corr_pairs = high_corr_pairs.sort_values("corr_abs", ascending=False).reset_index(drop=True)

print(f"Pares com |corr| >= {CORR_THRESHOLD}: {len(high_corr_pairs)}")
display(high_corr_pairs.head(50))

# Matriz (recorte) para inspecao rapida sem dependencia grafica extra.
print("Recorte 10x10 da matriz de correlacao (inicio da ordenacao de colunas):")
display(corr_matrix.iloc[:10, :10])


# 7. Diagnostico de redundancia por blocos tematicos

Objetivo:
- organizar features em blocos conceituais para apoiar futuros testes de ablation
- identificar blocos com sinais de redundancia interna

Regra adotada:
- inferencia por palavras-chave no nome da coluna
- classificacao diagnostica, nao definitiva

In [ ]:
def assign_block(feature_name: str) -> str:
    f = feature_name.lower()
    if any(k in f for k in ["idade", "sexo", "etnia", "civil", "membro", "sipra", "assentado"]) or feature_name in {"A", "AC", "B", "V", "Desconhecido"}:
        return "demografia"
    if any(k in f for k in ["renda", "produto", "atividade", "share", "tipos_renda", "possui_renda"]):
        return "renda_atividade"
    if any(k in f for k in ["ufpa", "modulo", "incra", "geo_valida", "soma_area", "soma_volume"]):
        return "produtivo_ufpa"
    if any(k in f for k in ["imovel", "n_imoveis", "lat_", "lon_", "regiao", "cd_uf"]):
        return "territorio_imoveis"
    return "outros"


block_map_df = pd.DataFrame({"feature": numeric_feature_cols})
block_map_df["bloco"] = block_map_df["feature"].map(assign_block)

block_rows: list[dict] = []
for bloco, feats in block_map_df.groupby("bloco")["feature"]:
    feats = list(feats)
    sub = df[feats]
    sub_corr = sub.corr(method="pearson", numeric_only=True)
    upper = sub_corr.where(np.triu(np.ones(sub_corr.shape, dtype=bool), k=1))
    abs_vals = stack_non_na(upper.abs())

    if len(abs_vals) > 0:
        max_abs_corr = float(abs_vals.max())
        mean_abs_corr = float(abs_vals.mean())
        n_pairs_high = int((abs_vals >= CORR_THRESHOLD).sum())
    else:
        max_abs_corr = np.nan
        mean_abs_corr = np.nan
        n_pairs_high = 0

    block_rows.append(
        {
            "bloco": bloco,
            "n_features": len(feats),
            "pct_missing_medio": float(sub.isna().mean().mean()),
            "max_abs_corr_interno": max_abs_corr,
            "mean_abs_corr_interno": mean_abs_corr,
            "n_pares_corr_alta_interno": n_pairs_high,
        }
    )

block_summary_df = pd.DataFrame(block_rows).sort_values("n_features", ascending=False).reset_index(drop=True)
print("Resumo por bloco tematico:")
display(block_summary_df)

print("Features por bloco (amostra):")
display(block_map_df.sort_values(["bloco", "feature"]).groupby("bloco").head(12).reset_index(drop=True))


# 8. Diagnostico de relacao das features com o alvo observado

Sem fazer selecao formal, avaliamos sinal univariado em relacao a `s`:
- correlacao simples feature vs `s`
- AUC univariada opcional para as variaveis com maior |corr|

Interpretacao:
- mede associacao com `s` observado, nao causalidade em `y` latente
- nao substitui avaliacao PU
- serve para detectar variaveis com sinal muito fraco ou potencialmente util

In [ ]:
import warnings

target_s = df[TARGET_COL].astype("int8")
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    corr_with_s = df[numeric_feature_cols].corrwith(target_s).rename("corr_with_s")

corr_target_df = (
    corr_with_s.to_frame()
    .reset_index()
    .rename(columns={"index": "feature"})
    .assign(corr_abs=lambda x: x["corr_with_s"].abs())
    .sort_values("corr_abs", ascending=False)
    .reset_index(drop=True)
)

print("Top 20 features por |corr(feature, s)|:")
display(corr_target_df.head(20))

top_auc_features = list(corr_target_df.head(15)["feature"])
auc_rows = []
for col in top_auc_features:
    x = pd.to_numeric(df[col], errors="coerce")
    if x.notna().sum() == 0 or x.nunique(dropna=True) <= 1:
        auc = np.nan
    else:
        x_filled = x.fillna(float(x.median()))
        auc = float(roc_auc_score(target_s, x_filled))

    auc_rows.append(
        {
            "feature": col,
            "auc_univariada": auc,
            "auc_orientada": (max(auc, 1.0 - auc) if pd.notna(auc) else np.nan),
        }
    )

auc_univariate_df = pd.DataFrame(auc_rows).sort_values("auc_orientada", ascending=False).reset_index(drop=True)
print("AUC univariada (top features por correlacao com s):")
display(auc_univariate_df)


# 9. Resumo dos achados

Consolidamos os principais sinais diagnosticos:
- quantidade de features com baixa variancia e concentracao extrema
- quantidade de pares altamente correlacionados
- blocos com possivel redundancia interna
- variaveis com maior sinal univariado em `s`

In [ ]:
n_zero_var = int(low_var_df["zero_variance"].sum())
n_candidate_review = int(low_var_df["candidate_review"].sum())
n_high_corr_pairs = int(len(high_corr_pairs))
n_blocks_redundancy = int((block_summary_df["n_pares_corr_alta_interno"] > 0).sum())

top_review_features = low_var_df.loc[low_var_df["candidate_review"], "feature"].head(15).tolist()
top_signal_features = corr_target_df["feature"].head(10).tolist()

summary_df = pd.DataFrame(
    [
        {"indicador": "n_features_total", "valor": len(feature_cols)},
        {"indicador": "n_features_numericas", "valor": len(numeric_feature_cols)},
        {"indicador": "n_zero_variance", "valor": n_zero_var},
        {"indicador": "n_candidates_low_variance_review", "valor": n_candidate_review},
        {"indicador": f"n_pairs_abs_corr_ge_{CORR_THRESHOLD}", "valor": n_high_corr_pairs},
        {"indicador": "n_blocos_com_corr_alta_interna", "valor": n_blocks_redundancy},
    ]
)
display(summary_df)

print("Top features candidatas a revisao (baixa variancia/concentracao):")
print(top_review_features)
print("\nTop features com maior sinal em s (diagnostico univariado):")
print(top_signal_features)


# 10. Recomendacao pratica para o projeto

Esta secao responde de forma objetiva:
- e necessario fazer higiene pesada?
- basta limpeza leve e revisao focal?
- vale manter todas as features no baseline e testar ablation por blocos?

In [ ]:
# Heuristica simples para transformar diagnostico em recomendacao pratica.
if n_zero_var >= 3 or n_high_corr_pairs >= 25:
    intensidade_higiene = "moderada"
elif n_candidate_review > 0 or n_high_corr_pairs > 0:
    intensidade_higiene = "leve"
else:
    intensidade_higiene = "minima"

if intensidade_higiene == "moderada":
    recomendacao_texto = (
        "Ha sinais relevantes de redundancia/baixa variancia. "
        "Recomenda-se limpeza orientada por diagnostico e testes de ablation por blocos."
    )
elif intensidade_higiene == "leve":
    recomendacao_texto = (
        "Nao ha indicio de higiene pesada. "
        "Recomenda-se limpeza leve, revisao manual das candidatas e manter baseline com todas as features."
    )
else:
    recomendacao_texto = (
        "Conjunto aparentemente saudavel. "
        "Manter features atuais e seguir com analise de importancia/ablation incremental."
    )

recomendacao_df = pd.DataFrame(
    [
        {"pergunta": "Necessita higiene pesada?", "resposta": "Nao" if intensidade_higiene != "moderada" else "Possivelmente sim"},
        {"pergunta": "Limpeza leve e recomendada?", "resposta": "Sim"},
        {"pergunta": "Colunas para revisao manual?", "resposta": ", ".join(top_review_features[:8]) if top_review_features else "Nenhuma obvia"},
        {"pergunta": "Seguir com todas as features no baseline?", "resposta": "Sim"},
        {"pergunta": "Executar ablation por bloco?", "resposta": "Sim, como proximo passo metodologico"},
    ]
)

print("Recomendacao geral:")
print(recomendacao_texto)
display(recomendacao_df)


# 11. Proximos passos

Passos sugeridos apos este diagnostico:
1. manter baseline com todas as features para comparabilidade historica
2. revisar manualmente o pequeno conjunto de candidatas apontadas
3. executar ablation por blocos tematicos (demografia, renda_atividade, produtivo_ufpa, territorio_imoveis)
4. comparar impacto em AUC_s e Recall@k sem mudar o protocolo PU
5. documentar decisoes finais de higiene somente apos evidencias de desempenho e estabilidade

In [ ]:
next_steps_df = pd.DataFrame(
    {
        "ordem": [1, 2, 3, 4, 5],
        "proximo_passo": [
            "Manter baseline com todas as features",
            "Revisar candidatas de baixa variancia/concentracao",
            "Rodar ablation por blocos tematicos",
            "Comparar metricas operacionais mantendo protocolo PU",
            "Formalizar decisoes de higiene com evidencias",
        ],
    }
)
display(next_steps_df)
